# Simpson's Paradox Is Not a Paradox

Every subgroup can move one way while the pooled data moves the other. This
is usually presented as a curiosity. It is better understood as a warning
about what a regression line is answering, because the arithmetic is not in
dispute and both lines are correct.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260903)

# Two departments. The second admits more able applicants and grades harder.
# Within each, more coaching lowers the score a little; across the two, the
# heavily coached department looks better.
n = 120
coaching_a = rng.uniform(0, 4, n)
coaching_b = rng.uniform(6, 10, n)
score_a = 70 - 1.2 * coaching_a + rng.normal(0, 2.0, n)
score_b = 88 - 1.2 * coaching_b + rng.normal(0, 2.0, n)

coaching = np.concatenate([coaching_a, coaching_b])
score = np.concatenate([score_a, score_b])
group = np.array(['A'] * n + ['B'] * n)

print(f'{len(coaching)} applicants across two departments')

## The pooled picture

Fit one line to everything and coaching looks beneficial.

In [ ]:
pooled_slope, pooled_intercept = np.polyfit(coaching, score, 1)

figure, axis = plt.subplots(figsize=(7, 4.5))
axis.scatter(coaching, score, s=18, color='#273338', alpha=0.55)
grid = np.linspace(coaching.min(), coaching.max(), 100)
axis.plot(grid, pooled_intercept + pooled_slope * grid,
          color='#c0392b', linewidth=2.5, label=f'pooled: {pooled_slope:+.2f}')
axis.set(xlabel='hours of coaching', ylabel='exam score')
axis.legend()
plt.show()

print(f'pooled slope: {pooled_slope:+.3f} points per hour')

The slope is positive and it is not a rounding error. Taken at face value,
the advice writes itself: coach the applicants.

## The same points, split

Now colour by department and fit each one separately.

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4.5))
colours = {'A': '#2b5748', 'B': '#9cb080'}

for name in ['A', 'B']:
    mask = group == name
    slope, intercept = np.polyfit(coaching[mask], score[mask], 1)
    axis.scatter(coaching[mask], score[mask], s=18, alpha=0.65,
                 color=colours[name], label=f'dept {name}: {slope:+.2f}')
    span = np.linspace(coaching[mask].min(), coaching[mask].max(), 50)
    axis.plot(span, intercept + slope * span, color=colours[name], linewidth=2.5)

axis.plot(grid, pooled_intercept + pooled_slope * grid, color='#c0392b',
          linewidth=2.5, linestyle='--', label=f'pooled: {pooled_slope:+.2f}')
axis.set(xlabel='hours of coaching', ylabel='exam score')
axis.legend()
plt.show()

Both departments slope down. The pooled line slopes up. Nothing has been
resampled and no observation has moved; the only change is which question
the line was asked.

## Where the sign comes from

The pooled slope mixes two quantities that have nothing to do with each
other: the within-department effect of coaching, and the between-department
difference in baseline. Department B coaches more and scores higher, and the
pooled line reads that coincidence as a causal slope.

In [ ]:
rows = []
for name in ['A', 'B']:
    mask = group == name
    slope, _ = np.polyfit(coaching[mask], score[mask], 1)
    rows.append((name, mask.sum(), coaching[mask].mean(), score[mask].mean(), slope))

print(f"{'dept':<6}{'n':>5}{'mean coaching':>16}{'mean score':>13}{'slope':>9}")
for name, count, mean_coaching, mean_score, slope in rows:
    print(f'{name:<6}{count:>5}{mean_coaching:>16.2f}{mean_score:>13.2f}{slope:>9.2f}')
print(f"{'pooled':<6}{len(coaching):>5}{coaching.mean():>16.2f}{score.mean():>13.2f}{pooled_slope:>9.2f}")

## What to do about it

The reversal is not evidence that pooling is wrong and splitting is right.
Splitting on a variable that is itself a consequence of the treatment
introduces a bias of its own, and the same picture can be produced that way
round. The question is whether department is a confounder or a mediator, and
no amount of staring at the scatterplot will answer it.

That answer comes from knowing how the data were made. A paradox that
dissolves once you say what caused what was never a paradox.